# 04 — Compare Formats

Run all prompt variants (current bullets, mini recap, V2 narrative) on the same
transcript and compare results side by side.

In [1]:
# Parameters (papermill-compatible)
book_slug = "hp-book1-ss"
chapter_index = 2
position_seconds = -1  # -1 = end of chapter
window_minutes = 5.0
variants = []  # Empty = all variants

In [2]:
import sys, os
# Ensure lib/ is importable regardless of CWD
_eval_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if _eval_root not in sys.path:
    sys.path.insert(0, _eval_root)

from pathlib import Path
from lib.transcript import (
    load_fixture, load_chapters, format_transcript, select_window
)
from lib.prompts import PROMPTS
from lib.token_budget import estimate_tokens
from lib.fm_client import check_availability
from lib.evaluate import run_comparison, display_side_by_side, check_grounding

In [3]:
available, reason = check_availability()
assert available, f"FM not available: {reason}"
print(f"FM available: {reason}")

FM available: Available


## Load transcript and select window

In [4]:
fixture_dir = Path(f"../fixtures/{book_slug}")
chapters = load_chapters(fixture_dir / "chapters.json")
ch = chapters[chapter_index]

all_sentences = load_fixture(fixture_dir / f"transcript_ch{chapter_index:02d}.json")
pos = position_seconds if position_seconds >= 0 else all_sentences[-1].end_time
window = select_window(all_sentences, position=pos, window_minutes=window_minutes)

transcript_text = format_transcript(window)
print(f"Chapter: {ch.title}")
print(f"Position: {pos:.0f}s | Window: {window_minutes} min | Sentences: {len(window)}")
print(f"Transcript tokens: ~{estimate_tokens(transcript_text)}")

Chapter: Chapter Two - The Vanishing Glass
Position: 3315s | Window: 5.0 min | Sentences: 62
Transcript tokens: ~931


## Run all variants

In [5]:
variant_list = variants if variants else list(PROMPTS.keys())
print(f"Running {len(variant_list)} variants: {variant_list}\n")

results_df = await run_comparison(window, variant_list)
display_side_by_side(results_df)

Running 6 variants: ['current_bullets', 'current_mini_recap_default', 'current_mini_recap_sleep', 'current_mini_recap_interrupted', 'v2_narrative', 'v2_narrative_simple']



                                             Prompt Comparison Results                                             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                           ┃ Tokens         ┃          ┃                                                         ┃
┃ Variant                   ┃ (in/out)       ┃ Latency  ┃ Response                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ current_bullets           │ 1092/176       │ 5.33s    │ - • Harry observes a Boa Constrictor at the zoo, noting │
│                           │                │          │ it's bred there.                                        │
│                           │                │          │ - • Dudley interrupts Harry, causing Harry to fall and  │
│                           │                │          │ nearly miss the snake.                                  │
│                           │                │          │ - • The Boa Constrictor's glass tank suddenly dis...    │
├───────────────────────────┼────────────────┼──────────┼─────────────────────────────────────────────────────────┤
│ current_mini_recap_defau… │ 1063/105       │ 2.64s    │ Harry witnessed an incredible event at the zoo: the     │
│                           │                │          │ glass in a Boa Constrictor tank vanished, and the snake │
│                           │                │          │ slithered out onto the floor, sending everyone into a   │
│                           │                │          │ panic. Dudley and Piers were horrified...               │
├───────────────────────────┼────────────────┼──────────┼─────────────────────────────────────────────────────────┤
│ current_mini_recap_sleep  │ 1074/94        │ 2.46s    │ Harry witnessed something incredible at the zoo: a Boa  │
│                           │                │          │ Constrictor escaped its tank, slithering out onto the   │
│                           │                │          │ floor and even managed to say "Brazil, here I come"     │
│                           │                │          │ before disappearing into the crowd. It ...              │
├───────────────────────────┼────────────────┼──────────┼─────────────────────────────────────────────────────────┤
│ current_mini_recap_inter… │ 1070/104       │ 2.61s    │ Harry was startled by Dudley and Piers as they leaned   │
│                           │                │          │ close to the glass of a Boa Constrictor. In the blink   │
│                           │                │          │ of an eye, the glass had vanished and the snake was     │
│                           │                │          │ slithering out of the tank! The snake hi...             │
├───────────────────────────┼────────────────┼──────────┼─────────────────────────────────────────────────────────┤
│ v2_narrative              │ 1182/154       │ 3.58s    │ ```json                                                 │
│                           │                │          │ {                                                       │
│                           │                │          │   "headline": "Snake escapes from glass tank",          │
│                           │                │          │   "recapNarrative": "Harry was just observing a Boa     │
│                           │                │          │ Constrictor when Dudley and Piers burst in, startling   │
│                           │                │          │ Harry. Suddenly, the glass vanished, ...                │
├───────────────────────────┼────────────────┼──────────┼─────────────────────────────────────────────────────────┤
│ v2_narrative_simple       │ 1133/100       │ 2.3s     │ ```json                                                 │
│                           │                │          

## Full responses

In [6]:
for _, row in results_df.iterrows():
    print(f"\n{'='*60}")
    print(f"VARIANT: {row['variant']}")
    print(f"Tokens: {row['input_tokens']} in / {row['output_tokens']} out | Latency: {row['latency_s']}s")
    print(f"{'='*60}")
    if row['error']:
        print(f"ERROR: {row['error']}")
    else:
        print(row['response'])


VARIANT: current_bullets
Tokens: 1092 in / 176 out | Latency: 5.33s
- • Harry observes a Boa Constrictor at the zoo, noting it's bred there.
- • Dudley interrupts Harry, causing Harry to fall and nearly miss the snake.
- • The Boa Constrictor's glass tank suddenly disappears, and the snake escapes.
- • Visitors scream and flee as the snake passes, muttering "Brazil, here I come."
- • The zoo staff, including the director, are bewildered by the glass vanishing.
- • Dudley exaggerates the snake's attack on him, while Piers claims it tried to kill him.
- • Uncle Vernon punishes Harry with a harsh warning in his cupboard.
- • Harry reflects on his past, haunted by visions of his parents and strange encounters.

Where we are now: Harry is confined in his cupboard, grappling with memories of his parents and feeling isolated from his family and peers.

VARIANT: current_mini_recap_default
Tokens: 1063 in / 105 out | Latency: 2.64s
Harry witnessed an incredible event at the zoo: the glass in a

## Grounding analysis

In [7]:
for _, row in results_df.iterrows():
    if row['error']:
        continue
    print(f"\n--- {row['variant']} ---")
    grounding = check_grounding(row['response'], transcript_text)
    grounded_count = sum(1 for g in grounding if g['grounded'])
    print(f"Grounded: {grounded_count}/{len(grounding)}")
    for g in grounding:
        status = "✓" if g['grounded'] else "✗"
        print(f"  {status} [{g['best_match_ratio']:.2f}] {g['sentence'][:70]}")


--- current_bullets ---
Grounded: 0/9
  ✗ [0.48] • Harry observes a Boa Constrictor at the zoo, noting it's bred there.
  ✗ [0.43] • Dudley interrupts Harry, causing Harry to fall and nearly miss the s
  ✗ [0.48] • The Boa Constrictor's glass tank suddenly disappears, and the snake 
  ✗ [0.49] • Visitors scream and flee as the snake passes, muttering "Brazil, her
  ✗ [0.36] • The zoo staff, including the director, are bewildered by the glass v
  ✗ [0.50] • Dudley exaggerates the snake's attack on him, while Piers claims it 
  ✗ [0.47] • Uncle Vernon punishes Harry with a harsh warning in his cupboard.
  ✗ [0.41] • Harry reflects on his past, haunted by visions of his parents and st
  ✗ [0.43] Where we are now: Harry is confined in his cupboard, grappling with me

--- current_mini_recap_default ---
Grounded: 0/1
  ✗ [0.09] Harry witnessed an incredible event at the zoo: the glass in a Boa Con

--- current_mini_recap_sleep ---
Grounded: 0/1
  ✗ [0.16] Harry witnessed something incredibl

## Token usage comparison

In [8]:
summary = results_df[['variant', 'input_tokens', 'output_tokens', 'total_tokens', 'latency_s', 'response_chars']].copy()
summary = summary.set_index('variant')
print(summary.to_string())

                                input_tokens  output_tokens  total_tokens  latency_s  response_chars
variant                                                                                             
current_bullets                         1092            176          1268       5.33             788
current_mini_recap_default              1063            105          1168       2.64             482
current_mini_recap_sleep                1074             94          1168       2.46             443
current_mini_recap_interrupted          1070            104          1174       2.61             435
v2_narrative                            1182            154          1336       3.58             688
v2_narrative_simple                     1133            100          1233       2.30             382
